<h1>Chapter 8 - Agentic RAG</h1>
<i>Building a multi-agent system using OpenAI SDK.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch08_agentic_rag/02_OpenAI_SDK/building_agents_with_openai_sdk.ipynb)

---

This notebook is for Chapter 8 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Building a multi-agent system using OpenAI SDK

In [8]:
!pip install openai-agents
!pip install pdf2image
!pip install chromadb

In [9]:
# Load sample data from GitHub
import requests

url = "https://raw.githubusercontent.com/polzerdo55862/rag-oreily-book/main/code/10_Agentic_RAG/10_02_OpenAI_SDK/sample_data/sample_email_history.txt"
response = requests.get(url)

with open("./sample_data/sample_email_history.txt", "w", encoding="utf-8") as f:
    f.write(response.text)
    
print(f"Successfully downloaded sample_email_history.txt from GitHub")

Successfully downloaded sample_email_history.txt from GitHub


In [21]:
# tag::define_pdf_file_browser_tool[]
import os
from typing import List
from agents import function_tool

@function_tool
def list_pdf_files(directory: str) -> List[str]:
    """Scan the given directory and return a list of PDF file paths."""
    pdf_paths = [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.lower().endswith('.pdf')
    ]
    print(f"Found {len(pdf_paths)} PDF files in directory '{directory}'.")
    return pdf_paths
# end::define_pdf_file_browser_tool[]

In [23]:
from agents import Agent

# tag::define_first_openai_agents_agent[]

agent = Agent(
    name="Assistant",
    tools=[list_pdf_files],
)
# end::define_first_openai_agents_agent[]

In [24]:
# tag::define_research_assistant_agent[]
from agents import Agent, Runner, WebSearchTool

# Create a research assistant with web search capability
research_assistant = Agent(
   name="Research Assistant",
   instructions="""You are a research assistant that helps users find and 
   summarize information.

   When asked about a topic:

   1. Search the web for relevant, up-to-date information
   2. Synthesize the information into a clear, concise summary
   3. Structure your response with headings and bullet points when appropriate
   4. Always cite your sources at the end of your response

   If the information might be time-sensitive or rapidly changing, mention when
   the search was performed.
   """,
   tools=[WebSearchTool()]
)

async def research_topic(topic):
   result = await Runner.run(research_assistant, f"Please research and summarize: {topic}. Only return the found links with very minimal text.")
   return result.final_output
# end::define_research_assistant_agent[]

In [25]:
# Usage example (in Jupyter notebook)
summary = await research_topic("From what you know, would you buy Apple stocks at the moment?")
print(summary)

Error getting response: Connection error.. (request_id: None)


APIConnectionError: Connection error.

[non-fatal] Tracing: request failed: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)


In [ ]:
# tag::import_all_emails_to_chromadb[]
import os
import shutil
import chromadb
from chromadb.config import Settings

def fill_chroma_db():
    chroma_dir = "./chroma_email_history_db"
    # If the directory exists, remove it to overwrite
    if os.path.exists(chroma_dir):
        shutil.rmtree(chroma_dir)
    client = chromadb.Client(Settings(persist_directory=chroma_dir))
    collection = client.get_or_create_collection("email_history")

    with open("./sample_data/sample_email_history.txt", "r", encoding="utf-8") as f:
        email_text = f.read()

    email_docs = [e.strip() for e in email_text.split("\n---\n") if e.strip()]

    for idx, doc in enumerate(email_docs):
        collection.add(documents=[doc], ids=[f"email_{idx+1}"])
        print(f"Added email_{idx+1}")

fill_chroma_db()
# end::import_all_emails_to_chromadb[]

Added email_1


In [26]:
# import helper_functions_agents_sdk
# from importlib import reload
# reload(helper_functions_agents_sdk)

# helper_functions_agents_sdk.query_emails("I give you 500 bucks")

[non-fatal] Tracing: request failed: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)


In [28]:
# tag::define_query_database_tool[]
import chromadb
from chromadb.config import Settings

@function_tool
def query_database(query_text: str, n_results: int = 3) -> dict:
    """
    Query emails using semantic search from a local ChromaDB collection.
    """

    # Set up ChromaDB client and collection
    chroma_dir = "./chroma_email_history_db"
    client = chromadb.Client(Settings(persist_directory=chroma_dir))
    collection = client.get_or_create_collection("email_history")

    # Perform the query
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results,
        include=["documents"],
    )
    return {
        "query": query_text,
        "documents": results["documents"][0] if results["documents"] else [],
    }
# end::define_query_database_tool[]

In [31]:
# tag::define_negotiation_agent[]
from agents import Agent, Runner
from agents import trace

# Create a negotiation agent with email search capability
salesman_agent = Agent(
    name="Negotiation Agent",
    instructions="""You are a skilled negotiation agent representing a salesperson.

    Your role:
    - Analyze the current email conversation history
    - Search for relevant information using the query_database tool
    - Generate informed responses with strategic counter-offers

    Process:
    1. Use query_database to find relevant context from email history
    2. Analyze customer concerns and pricing constraints
    3. Craft a professional response with a competitive counter-offer
    4. Maintain a collaborative tone while protecting profit margins
    """,
    tools=[query_database],
    model="gpt-4o"
)

# Helper function to run the negotiation agent
async def negotiate_with_customer(email_history: str) -> str:
    with trace("Negotiation Response"):
        result = await Runner.run(
            salesman_agent,
            f"Email conversation history:\n{email_history}"
        )
    return result.final_output
# end::define_negotiation_agent[]

In [34]:
# tag::use_negotiation_agent[]
# Example: Using the negotiation agent
import asyncio
from agents import trace

async def demonstrate_negotiation():
    with open("./sample_data/sample_email_history.txt",
              "r", encoding="utf-8") as f:
        email_history = f.read()

    # Generate negotiation response using the agent
    with trace("Automated Customer Response"):
        response = await negotiate_with_customer(email_history)

    print(response)
    return response

await demonstrate_negotiation()
# end::use_negotiation_agent[]

Trace already exists. Creating a new trace, but this is probably a mistake.
Error getting response: Connection error.. (request_id: None)


APIConnectionError: Connection error.

[non-fatal] Tracing: request failed: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)
[non-fatal] Tracing: request failed: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)
[non-fatal] Tracing: request failed: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)
[non-fatal] Tracing: max retries reached, giving up on this batch.


In [ ]:
with open("./sample_data/sample_email_history.txt", "r", encoding="utf-8") as f:
    email_history = f.read()

email_history

In [ ]:
# tag::define_moderated_negotiation_process[]
from agents import Agent, Runner

async def run_negotiation_process():
    """
    Orchestrates a negotiation between customer and salesperson agents
    using a moderator agent to facilitate the conversation.
    """
    # Load the email history from file
    with open("./sample_data/sample_email_history.txt", "r", encoding="utf-8") as f:
        email_history = f.read()

    # Define the customer agent
    customer_agent = Agent(
        name="Customer",
        instructions="""Negotiate for the best laptop deal.
                       Be polite but persistent.
                       Respond to the email chain below.""",
        model="gpt-4o-mini",
    )

    # Convert agents to tools for the moderator
    customer_agent_tool = customer_agent.as_tool(
        tool_name="customer",
        tool_description="Tool that represents the customer in negotiations."
    )

    salesman_agent_tool = salesman_agent.as_tool(
        tool_name="salesman",
        tool_description="Tool that represents the salesperson in negotiations."
    )

    # Define the moderator agent with clear instructions
    moderator_instructions = """
    You are a moderator facilitating negotiation between customer and salesperson.

    Process:
    1. Receive email history with the last message from customer
    2. Use salesman_agent_tool to generate salesperson response
    3. Append response to email history
    4. Use customer_agent_tool to generate customer response
    5. Continue alternating until agreement or breakdown

    Rules:
    - Aim for mutually beneficial agreement
    """

    moderator_agent = Agent(
        name="Moderator",
        instructions=moderator_instructions,
        model="gpt-4o-mini",
        tools=[customer_agent_tool, salesman_agent_tool]
    )

    # Execute the negotiation with tracing
    with trace("Negotiation Process"):
        responses = await Runner.run(
            moderator_agent,
            f"Begin negotiation with this email history: {email_history}"
        )

    return responses
# end::define_moderated_negotiation_process[]

In [ ]:
responses = await run_negotiation_process()

In [ ]:
# tag::define_direct_negotiation_process[]
from agents import Agent, Runner

async def run_negotiation_process_with_handoffs():
    """
    Direct negotiation between customer and salesperson agents
    using handoffs to alternate between them.
    """
    # Load the email history from file
    with open("./sample_data/sample_email_history.txt", "r", encoding="utf-8") as f:
        email_history = f.read()

    # Define the salesperson agent
    salesman_agent = Agent(
        name="Salesperson",
        instructions="""You are a laptop salesperson negotiating a deal.
                       Make reasonable offers and try to close the sale.
                       If you reach agreement, clearly state 'DEAL AGREED'.
                       If negotiation isn't working after 3 rounds, politely end it.""",
        model="gpt-4o-mini",
        handoff_description="Hand off to customer for their response to your offer."
    )

    # Define the customer agent with handoff capabilities
    customer_agent = Agent(
        name="Customer",
        instructions="""Negotiate for the best laptop deal.
                       Be polite but persistent.
                       Make counteroffers or accept good deals.
                       If you accept a deal, clearly state 'DEAL ACCEPTED'.
                       If you can't reach agreement after 3 rounds, politely walk away.""",
        model="gpt-4o-mini",
        handoff_description="Hand off to salesperson for their response to your counteroffer."
    )

    # Convert each agent to a tool for the other
    customer_tool = customer_agent.as_tool(
        tool_name="handoff_to_customer",
        tool_description="Hand the conversation to the customer agent."
    )

    salesman_tool = salesman_agent.as_tool(
        tool_name="handoff_to_salesperson",
        tool_description="Hand the conversation to the salesperson agent."
    )

    # Give each agent the ability to hand off to the other
    customer_agent.tools = [salesman_tool]
    salesman_agent.tools = [customer_tool]

    # Start with the salesperson responding to the email history
    with trace("Direct Negotiation Process"):
        responses = await Runner.run(
            salesman_agent,
            f"Respond to this email history and begin negotiation: {email_history}"
        )

    return responses
# end::define_direct_negotiation_process[]

In [ ]:
responses = await run_negotiation_process_with_handoffs()

In [ ]:
responses